# 01 — Exploration du corpus Allociné

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

import pandas as pd
from src.config import load_config

cfg = load_config()
df = pd.read_parquet(ROOT / "data/raw/allocine_5000.parquet")

## 1. Aperçu

In [2]:
df.sample(10, random_state=cfg["seed"])

,review,label
1501,"""Plaît-il ??"" Trop drôle. J'ai vraiment beauco...",1
2586,"Un film super drôle , extrêmement bien joué . ...",1
2653,Trois étoiles TRES généreuses. J'ai été très d...,1
1055,J'ai découvert Robert Rodriguez avec Sin City(...,1
705,Autant j'adore John Turturro en tant qu'acteur...,0
106,J'ai mis 2 étoiles pour l'ambiance du film qui...,0
589,Un film a petit budget qui arrive à confondre ...,1
2468,"Brillant, atmosphère subtile. Polar réussit et...",1
2413,Film d'animation correcte mais que l'on oublie...,0
1600,"J’ai vu ce film quand j’étais enfant, et je me...",0


## 2. Distribution des labels

In [3]:
df.label.value_counts(normalize=True)

label
0    0.5076
1    0.4924
Name: proportion, dtype: float64

**Conclusion** : corpus équilibré (50,8 % négatifs, 49,2 % positifs).

## 3. Longueur des critiques

In [4]:
df["n_chars"] = df.review.str.len()
df["n_words"] = df.review.str.split().str.len()
df.groupby("label")[["n_chars", "n_words"]].describe()

n_chars                                                              \
        count        mean         std  min     25%    50%     75%     max   
label                                                                       
0      2538.0  521.671789  434.334017  4.0  188.25  379.0  736.75  2000.0   
1      2462.0  566.570674  464.530713  7.0  202.00  408.5  810.50  2000.0   

      n_words                                                       
        count       mean        std  min   25%   50%    75%    max  
label                                                               
0      2538.0  88.582348  73.239569  1.0  33.0  65.0  125.0  361.0  
1      2462.0  94.846872  77.296159  2.0  35.0  69.0  136.0  353.0

In [5]:
(df.n_words < 5).mean()

np.float64(0.0064)

In [6]:
from sentence_transformers import SentenceTransformer

emb = cfg["embedding"]
model = SentenceTransformer(emb["model"])
df["n_model_tokens"] = [len(model.tokenizer.encode(emb["prefix"] + t)) for t in df["review"]]
(df["n_model_tokens"] > model.max_seq_length).mean()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (541 > 512). Running this sequence through the model will result in indexing errors


np.float64(0.0112)

In [7]:
import plotly.express as px

px.histogram(
    df,
    x="n_words",
    color="label",
    barmode="overlay",
    nbins=60,
    labels={"n_words": "Nombre de mots", "label": "Label"},
)

## 4. Bruit dans les textes

In [8]:
import re

patterns = {
    "html_tag": r"<[^>]+>",
    "html_entity": r"&[a-z]+;|&#\d+;",
    "url": r"https?://\S+|www\.\S+",
    "apostrophe_typo": r"[’‘`´]",
    "guillemets": r"[«»“”]",
    "ponct_repetee": r"[!?.]{3,}",
    "lettres_repetees": r"([a-zA-ZÀ-ÿ])\1{2,}",
    "mot_majuscules": r"\b[A-ZÀ-Ý]{4,}\b",
    "emoji": r"[\U0001F300-\U0001FAFF\u2600-\u27BF]",
    "note_chiffree": r"\b\d{1,2}\s?/\s?\d{1,2}\b",
    "espaces_multiples": r"\s{2,}",
    "retour_ligne": r"\n",
}

In [9]:
rows = []
for name, pat in patterns.items():
    rx = re.compile(pat)
    mask = df["review"].map(lambda t: rx.search(t) is not None)
    rows.append({"motif": name, "n": mask.sum(), "part": mask.mean()})
noise = pd.DataFrame(rows).sort_values("part", ascending=False)
noise

,motif,n,part
5,ponct_repetee,1483,0.2966
3,apostrophe_typo,639,0.1278
4,guillemets,224,0.0448
7,mot_majuscules,191,0.0382
9,note_chiffree,114,0.0228
6,lettres_repetees,69,0.0138
2,url,20,0.0040
8,emoji,12,0.0024
1,html_entity,1,0.0002
0,html_tag,0,0.0000


In [10]:
def show(pat: str, k: int = 3) -> None:
    rx = re.compile(pat)
    hits = [t for t in df["review"] if rx.search(t)][:k]
    for t in hits:
        m = rx.search(t)
        print("…", t[max(0, m.start() - 40) : m.end() + 40], "…\n")

In [11]:
show(patterns["note_chiffree"])

… ite: http://www.leblogducinema.com/2010/09/02/critique-copains-pour-toujours/ …

… ironnement plutôt soigné, rien de bon ! 1/5 !!! …

… dommage que l'on ne puissent pas mettre 0/5 …



In [12]:
show(patterns["lettres_repetees"])

… iou Miou au top et Dewaere que je connaisssais pas est vraiment excellent aussi (su …

… icative. Dommage. Lire la suite: http://www.leblogducinema.com/2010/09/02/critique- …

… st la presence de Demi Moore et Bruce Willlis. …



## 5. Mots fréquents

In [13]:
import spacy
from collections import Counter

nlp = spacy.load(cfg["preprocessing"]["spacy_model"], disable=["parser"])

texts = df["review"].str.replace(r"[’‘`´]", "'", regex=True).tolist()

lemmas, persons = [], []
for doc in nlp.pipe(texts, batch_size=64):
    lemmas.append(
        [
            token.lemma_.lower()
            for token in doc
            if token.is_alpha and not token.is_stop and len(token) > 1
        ]
    )
    persons.append([e.text for e in doc.ents if e.label_ == "PER"])

df["lemmas"] = lemmas
df["persons"] = persons

In [14]:
for lab in (0, 1):
    c = Counter(l for doc in df.loc[df["label"] == lab, "lemmas"] for l in doc)
    print(lab, c.most_common(30))

0 [('film', 3509), ('bien', 1006), ('acteur', 830), ('bon', 729), ('scène', 671), ('faire', 662), ('voir', 657), ('histoire', 654), ('scénario', 625), ('rien', 604), ('trop', 518), ('personnage', 506), ('mal', 479), ('vraiment', 449), ('passer', 401), ('grand', 387), ('aucun', 363), ('long', 328), ('fin', 324), ('être', 321), ('chose', 310), ('petit', 307), ('mauvais', 305), ('réalisateur', 303), ('vouloir', 291), ('mettre', 279), ('non', 270), ('temps', 250), ('intérêt', 247), ('genre', 246)]
1 [('film', 4008), ('bien', 1222), ('bon', 984), ('voir', 740), ('histoire', 730), ('acteur', 724), ('scène', 719), ('grand', 638), ('personnage', 619), ('excellent', 433), ('petit', 431), ('faire', 402), ('scénario', 385), ('cinéma', 380), ('passer', 368), ('vie', 359), ('vraiment', 354), ('fois', 342), ('fin', 334), ('rôle', 320), ('moment', 319), ('être', 316), ('bel', 304), ('réalisateur', 296), ('meilleur', 286), ('oeuvre', 275), ('mise', 274), ('beaucoup', 266), ('magnifique', 260), ('beau'

In [15]:
doc_freq = Counter(l for doc in df["lemmas"] for l in set(doc))
top = pd.DataFrame(doc_freq.most_common(50), columns=["lemme", "n_docs"])
top["part_docs"] = (top["n_docs"] / len(df)).round(3)
top

,lemme,n_docs,part_docs
0,film,3657,0.731
1,bien,1566,0.313
2,acteur,1342,0.268
3,bon,1269,0.254
4,histoire,1155,0.231
5,voir,1134,0.227
6,scène,1079,0.216
7,scénario,954,0.191
8,personnage,893,0.179
9,faire,866,0.173


In [16]:
def doc_share(lab: int) -> pd.Series:
    sub = df.loc[df["label"] == lab, "lemmas"]
    c = Counter(l for doc in sub for l in set(doc))
    return pd.Series(c) / len(sub)


comp = pd.DataFrame({"neg": doc_share(0), "pos": doc_share(1)}).fillna(0)
comp["ecart"] = comp["pos"] - comp["neg"]
comp.sort_values("ecart").head(20)

,neg,pos,ecart
rien,0.194641,0.073924,-0.120718
mal,0.156422,0.058489,-0.097933
mauvais,0.101655,0.015435,-0.086220
scénario,0.231678,0.148660,-0.083019
trop,0.161939,0.081235,-0.080704
aucun,0.115445,0.042648,-0.072797
intérêt,0.089441,0.017872,-0.071569
faire,0.206462,0.138911,-0.067550
ridicule,0.060678,0.006093,-0.054585
ennuyeux,0.057132,0.002843,-0.054288


In [17]:
comp.sort_values("ecart").tail(20)

,neg,pos,ecart
voir,0.207644,0.246548,0.038904
parfaitement,0.008274,0.047522,0.039248
musique,0.034673,0.073924,0.039251
fois,0.074074,0.114947,0.040873
homme,0.034673,0.075548,0.040875
cinéma,0.078014,0.121852,0.043838
moment,0.067376,0.116572,0.049196
beau,0.042159,0.093826,0.051667
film,0.705674,0.757920,0.052247
oeuvre,0.038613,0.091389,0.052776


## 6. Noms propres

In [18]:
Counter(p for doc in df["persons"] for p in doc).most_common(30)

[('Sinon', 30),
 ('Scénario', 28),
 ('Spielberg', 26),
 ('Hitchcock', 23),
 ('James Bond', 23),
 ('Clint Eastwood', 20),
 ('John Wayne', 19),
 ('Woody Allen', 17),
 ('Godard', 17),
 ('Bruce Willis', 15),
 ('Eastwood', 14),
 ('Isabelle Huppert', 14),
 ('Magnifique', 14),
 ('Polanski', 14),
 ('Superbe', 13),
 ('Al Pacino', 13),
 ('Kurosawa', 12),
 ('Daniel Auteuil', 12),
 ('Ben', 12),
 ('Gérard Depardieu', 11),
 ('Depardieu', 11),
 ('Chris', 11),
 ('Chaplin', 11),
 ('Johnny Depp', 11),
 ('Pierre Richard', 11),
 ('Nicolas Cage', 11),
 ('Antonioni', 11),
 ('Jeff Bridges', 11),
 ('Kubrick', 10),
 ('Ken Loach', 10)]

In [19]:
(df["persons"].str.len() > 0).mean()

np.float64(0.5892)